# In Class Activity 4/7
## Insurance Dataset


In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import sweetviz as sv
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

In [10]:
df = pd.read_csv("insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [11]:
report = sv.analyze(df)
report.show_html('insurane_report.html')

                                             |          | [  0%]   00:00 -> (? left)

Report insurane_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Some of the trends in this dataset include a normally distributed BMI centered around 30, a right skewed distribution for insurance charges with most between $0 and $10k. Additionally most families have less than 3 children. Close to 80% of individuals do not smoke. 

There are large associations between charges and a person's smoking status. Additionally, BMI and region demonstrate strong associations, as well as age and charges.  

In [12]:
##encoding categorical variables
insurance_encoded = pd.get_dummies(df, drop_first = True).astype(int)
insurance_encoded.head()

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27,0,16884,0,1,0,0,1
1,18,33,1,1725,1,0,0,1,0
2,28,33,3,4449,1,0,0,1,0
3,33,22,0,21984,1,0,1,0,0
4,32,28,0,3866,1,0,1,0,0


In [15]:
# preparing the data for modeling
X = insurance_encoded.drop('charges', axis = 1)
y = insurance_encoded['charges']

# splittting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= 222)

#scaling the features (don't need for R but do for linear reg)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# baseline linear reg
baseline_lr = LinearRegression()
baseline_lr.fit(X_train_scaled, y_train)
y_pred_lr = baseline_lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

# baseline random forest model
baseline_rf = RandomForestRegressor(random_state=222)
baseline_rf.fit(X_train, y_train)
y_pred_rf = baseline_rf.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

# print baseline results

# print baseline results
print(f'Baseline Linear Regression MSE: {mse_lr:.2f}, R2: {r2_lr:.2f}')
print(f'Baseline Random Forest MSE: {mse_rf:.2f}, R2: {r2_rf:.2f}')


Baseline Linear Regression MSE: 34398163.11, R2: 0.80
Baseline Random Forest MSE: 23875113.99, R2: 0.86


Both the baseline linear regression and baseline random forest model have high r2 values, 0.8 and 0.86 respectively, indicating a good fit and high explainability. The r2 value indicates the model captures the overall pattern and variance of the dataset.

However, both models have very high mean squared error's, indicating not as strong model performance, and large prediction error.  

In [16]:
# CV with baselinemodels with r2 as the scoring metric
cv_score_lr = cross_val_score(baseline_lr, X_train_scaled, y_train, cv = 5, scoring = 'r2')
cv_score_rf = cross_val_score(baseline_rf, X_train, y_train, cv = 5, scoring = 'r2')
print(f'Baseline Linear Regression CV R2 Scores: {cv_score_lr}')
print(f'Baseline Random Forest CV R2 Scores: {cv_score_rf}')  

Baseline Linear Regression CV R2 Scores: [0.63029986 0.74267626 0.7471662  0.74490129 0.76911128]
Baseline Random Forest CV R2 Scores: [0.75504616 0.8477652  0.81271924 0.83410053 0.87861578]


In [17]:
# gridsearchCV for hyperparameter tuning of random forest
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'max_features': ['log2', 'sqrt']
}

grid_search_rf = GridSearchCV(estimator=baseline_rf, 
                              param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)    

print(f'Best RF Hyperparameters: {grid_search_rf.best_params_}')
print(f'Best RF CV R2 Score: {grid_search_rf.best_score_:.2f}')


Best RF Hyperparameters: {'max_depth': None, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 100}
Best RF CV R2 Score: 0.84


In [19]:
# examine top 20 ranked parameter combinations from grid search 
print("\nTop 20 GridSearchCV results:")
top20 = (
    pd.DataFrame(grid_search_rf.cv_results_)
    .sort_values('mean_test_score', ascending=False)
    .head(20)
    [['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 
      'param_max_features', 'mean_test_score']]
    .rename(columns=lambda x: x.replace('param_', ''))
)
print(top20)


Top 20 GridSearchCV results:
    n_estimators max_depth  min_samples_split max_features  mean_test_score
2            100      None                  5         log2         0.840086
18           100        20                  5         log2         0.840086
19           200        20                  5         log2         0.839328
3            200      None                  5         log2         0.839328
11           200        10                  5         log2         0.839243
9            200        10                  2         log2         0.838231
10           100        10                  5         log2         0.837884
8            100        10                  2         log2         0.837255
17           200        20                  2         log2         0.832628
1            200      None                  2         log2         0.832598
16           100        20                  2         log2         0.831194
0            100      None                  2         log2

I would choose a max_depth of 10, min_samples_split of 5 and log2 for max_features because these all appear the most. As for n_estimators, because there is a tie between both values 100 and 200, I would tune this parameter more and potentially even include more values to choose from.

While the mean_test_score isn't terrible, I would try to tune some other parameters a little more to try and get into the 0.9 range. 

In [20]:
# fit tuned random forest using best parameters
best_rf = RandomForestRegressor(
    **grid_search_rf.best_params_,
    random_state=222
)

best_rf.fit(X_train, y_train)

# predictions
y_train_pred = best_rf.predict(X_train)
y_test_pred = best_rf.predict(X_test)

# performance metrics
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

print(f"Training R2: {train_r2:.4f}")
print(f"Test R2: {test_r2:.4f}")
print(f"Training MSE: {train_mse:.4f}")
print(f"Test MSE: {test_mse:.4f}")

Training R2: 0.9384
Test R2: 0.8675
Training MSE: 8621246.4198
Test MSE: 22846040.6181


Yes, the model is slightly overfitting because both the r2 value and MSE got worse in the test dataset compared to the values for the training dataset. 